In [5]:
import torch
import os
os.environ["TRITON_INTERPRET"] = "0"
import softmax1
import torch.nn.functional as F

def softmax_backward(dy, y):
    dot = (dy * y).sum(dim=-1, keepdim=True)
    dx = y * (dy - dot)
    return dx

In [6]:
x = [[1,1,4], [1,2,1]]

def loss_fn(x):
    return torch.sum(x**2)

x_pt = torch.asarray(x, dtype=torch.float32, device="cuda", requires_grad=True)
y_pt = F.softmax(x_pt, dim=-1)
y_pt.retain_grad()
loss_pt = loss_fn(y_pt)
loss_pt.retain_grad()
loss_pt.backward()

x_tr = torch.asarray(x, dtype=torch.float32, device="cuda", requires_grad=True)
y_tr = softmax1.softmax_with_bw(x_tr)
y_tr.retain_grad()
loss_tr = loss_fn(y_tr)
loss_tr.retain_grad()
loss_tr.backward()

assert torch.allclose(x_tr.grad, x_pt.grad)


In [7]:
softmax_backward(y_tr.grad, y_tr)

tensor([[-0.0712, -0.0712,  0.1423],
        [-0.0889,  0.1779, -0.0889]], device='cuda:0', grad_fn=<MulBackward0>)

In [8]:
softmax1.softmax_bwd(y_tr.grad, y_tr)

tensor([[-0.0712, -0.0712,  0.1423],
        [-0.0889,  0.1779, -0.0889]], device='cuda:0')

In [10]:
x_pt.grad

tensor([[-0.0712, -0.0712,  0.1423],
        [-0.0889,  0.1779, -0.0889]], device='cuda:0')